# 02. NESO's benchmark and time alignment

**Where we are:** notebook 01 downloaded NESO's historic demand and settled on the target,
National Demand (`ND`), in MW, for each half-hour settlement period.

**What this notebook does:**

1. Downloads NESO's own day-ahead forecast, which is the benchmark to beat.
2. Builds correct timestamps from settlement periods, including clock-change days.
3. Checks that NESO's "outturn" is the same quantity as `ND`.
4. Measures how accurate NESO's forecast is. That number is the bar for the rest of the project.

**Why bother with a benchmark:** a forecast error of 700 MW means nothing without context.
NESO's published forecast is what the industry actually uses, so beating it, or failing to
and saying by how much, is a meaningful result.

**Background reading**
- [Day Ahead Half Hourly Demand Forecast Performance](https://www.neso.energy/data-portal/day-ahead-half-hourly-demand-forecast-performance): the dataset used here.
- [Settlement period (Elexon glossary)](https://www.elexon.co.uk/glossary/settlement-period/): the half-hour unit used across the GB market.


## Step 1. Set up

**What:** imports, paths and plot style.

**Why:** same layout as notebook 01. Raw downloads go to `data/raw`, cleaned tables to
`data/processed`. Both are kept out of git; the notebooks recreate them.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

BLUE, ORANGE = "#2a78d6", "#eb6834"
plt.rcParams.update({
    "figure.dpi": 90,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e4e4e0",
    "lines.linewidth": 2,
})


## Step 2. Download the benchmark

**What:** look up the dataset on NESO's data portal API and save its CSV.

**Why the API:** the file is updated every day, so the notebook should find the current
download link itself rather than hard-code it.

**Note:** the download link redirects, and `requests` follows redirects by default. A plain
`curl` without `-L` saves a "302 Found" HTML page instead of the data, which I found out the
hard way. The check after the download catches that.


In [2]:
API = "https://api.neso.energy/api/3/action/package_show"
BENCHMARK_FILE = RAW / "neso_day_ahead_forecast.csv"

if not BENCHMARK_FILE.exists():
    package = requests.get(
        API, params={"id": "day-ahead-half-hourly-demand-forecast-performance"}, timeout=60
    ).json()["result"]
    resource = package["resources"][0]
    print("resource:", resource["name"])
    response = requests.get(resource["url"], timeout=300)
    response.raise_for_status()
    BENCHMARK_FILE.write_bytes(response.content)

first_line = BENCHMARK_FILE.open(encoding="utf-8-sig").readline()
assert first_line.startswith("Month,Date"), "download is not the expected CSV"
print(f"{BENCHMARK_FILE.name}: {BENCHMARK_FILE.stat().st_size / 1e6:.1f} MB")


neso_day_ahead_forecast.csv: 9.7 MB


## Step 3. First look

**What:** load the file and read the columns.

**Why `encoding="utf-8-sig"`:** the file starts with a byte-order mark, an invisible character
some Windows tools add. Without this, the first column is named `\ufeffMonth` and code that
looks up `Month` fails.

| Column | Meaning |
|---|---|
| `Date`, `Settlement_Period` | the target half-hour, in GB local time |
| `Datetime` | a timestamp label; checked in Step 7 |
| `Demand_Forecast` | NESO's day-ahead forecast, MW |
| `Demand_Outturn` | the demand that actually happened, MW |
| `TRIAD_Avoidance_Estimate` | demand NESO thinks large users cut on likely TRIAD periods (see Step 8) |
| `TRIAD_Avoidance_Corrected_Demand_Outturn` | outturn with that cut added back |
| `APE`, `Absolute_Error` | NESO's own error figures |
| `Publish_Datetime` | when the forecast was published |


In [3]:
bench = pd.read_csv(BENCHMARK_FILE, encoding="utf-8-sig")
print(bench.shape)
print("dates:", bench["Date"].min(), "to", bench["Date"].max())
bench.head()


(98370, 11)
dates: 2021-04-01 to 2026-11-10


,Month,Date,Datetime,Settlement_Period,Demand_Forecast,Demand_Outturn,TRIAD_Avoidance_Estimate,TRIAD_Avoidance_Corrected_Demand_Outturn,APE,Absolute_Error,Publish_Datetime
0,Apr,2021-04-01,2021-04-01T00:30:00Z,1,21570,19970,0,19970,8.0,1600,2021-03-31T09:45:00Z
1,Apr,2021-04-01,2021-04-01T01:00:00Z,2,21130,20030,0,20030,5.5,1100,2021-03-31T09:45:00Z
2,Apr,2021-04-01,2021-04-01T01:30:00Z,3,21272,20499,0,20499,3.8,773,2021-03-31T09:45:00Z
3,Apr,2021-04-01,2021-04-01T02:00:00Z,4,21600,20940,0,20940,3.2,660,2021-03-31T09:45:00Z
4,Apr,2021-04-01,2021-04-01T02:30:00Z,5,21478,20682,0,20682,3.8,796,2021-03-31T09:45:00Z


## Step 4. Data checks

**What:** look for missing values, repeated half-hours and days with the wrong number of periods.

**Why:** a day has 48 periods, or 46 and 50 on the spring and autumn clock-change days (see
notebook 01). Anything else means a fault in the file or a day that is not finished yet.


In [4]:
KEY = ["Date", "Settlement_Period"]
print("missing values:", int(bench.isna().sum().sum()))

repeated = bench[bench.duplicated(KEY, keep=False)]
print("rows sharing a date and period:", len(repeated))
print(repeated[KEY + ["Datetime", "Demand_Outturn"]].to_string())


missing values: 0
rows sharing a date and period: 8
             Date  Settlement_Period              Datetime  Demand_Outturn
10225  2021-10-31                  4  2021-10-31T01:00:00Z           21087
10226  2021-10-31                  4  2021-10-31T01:00:00Z           21087
10227  2021-10-31                  5  2021-10-31T01:30:00Z           20743
10228  2021-10-31                  5  2021-10-31T01:30:00Z           20743
27697  2022-10-30                  2  2022-10-30T01:00:00Z           18302
27698  2022-10-30                  2  2022-10-30T01:00:00Z           18302
27699  2022-10-30                  3  2022-10-30T01:30:00Z           17778
27700  2022-10-30                  3  2022-10-30T01:30:00Z           17778


In [5]:
periods = bench.groupby("Date")["Settlement_Period"].agg(["count", "max"])
print(periods["count"].value_counts().sort_index().to_string())
print()
print(periods[periods["count"] != 48].to_string())


count
18       1
46       5
48    2039
50       5

            count  max
Date                  
2021-10-31     50   50
2022-03-27     46   46
2023-03-26     46   46
2023-10-29     50   50
2024-03-31     46   46
2024-10-27     50   50
2025-03-30     46   46
2025-10-26     50   50
2026-03-29     46   46
2026-10-25     50   50
2026-11-10     18   18


**Found:**

- No missing values.
- **Two autumn clock-change days are mislabelled.** On 2021-10-31 and 2022-10-30, periods 2
  and 3 are missing and periods 4 and 5 appear twice with the same outturn. The `Datetime`
  values (01:00, 01:30 twice) suggest the rows were matched on clock time, which repeats when
  the clocks go back. There is no safe way to tell which row is which, so I drop all of them.
  That loses 8 rows out of about 98,000.
- The other clock-change days have the right 46 or 50 periods.
- The last day is part-way through, because NESO adds outturn as the day goes on. I keep only
  complete days.


In [6]:
bench = bench[~bench.duplicated(KEY, keep=False)].copy()

per_day = bench.groupby("Date")["Settlement_Period"]
complete = (per_day.transform("count") == per_day.transform("max")) & (per_day.transform("max") >= 46)
print("dropped (incomplete days):", sorted(bench.loc[~complete, "Date"].unique()))
bench = bench[complete].reset_index(drop=True)
print(bench.shape, bench["Date"].min(), "to", bench["Date"].max())


dropped (incomplete days): ['2021-10-31', '2022-10-30', '2026-11-10']
(98254, 11) 2021-04-01 to 2026-11-09


## Step 5. When does NESO publish?

**What:** look at `Publish_Datetime`.

**Why:** this sets the information cut-off for our own forecast. If NESO publishes at 09:45 the
day before, our model must use only data available by then. Using anything later would give
us an unfair advantage and make the comparison worthless.


In [7]:
published = pd.to_datetime(bench["Publish_Datetime"], utc=True)
target_day = pd.to_datetime(bench["Date"]).dt.tz_localize("UTC")
hours_before = (target_day - published).dt.total_seconds() / 3600

print("published on the day before:", bool(((hours_before > 0) & (hours_before < 24)).all()))
print()
print(published.dt.strftime("%H:%M").groupby(published.dt.year).value_counts().rename("rows").to_string())


published on the day before: True

Publish_Datetime  Publish_Datetime
2021              09:45               10224
                  08:45                2976
2022              09:45               10368
                  08:45                7102
2023              09:45               10418
                  08:45                7102
2024              09:45               10082
                  08:45                7486
2025              09:45                8832
                  09:33                4466
                  08:45                4222
2026              09:33                8926
                  08:33                5666
                  10:03                 192
                  09:03                  96
                  10:33                  96


**Found:** every forecast is published on the day before the target day. The recorded times
run from 08:33 to 10:33, with two or more different times in every year. The values end in `Z`
(UTC), but Step 7 shows that the `Z` on `Datetime` is wrong, so I do not trust it here either.

**Decision:** our forecast origin is **08:30 UTC on the day before**, earlier than every
published time under any reading of the timestamps. If anything, this gives NESO a small
information advantage, which is the safe direction for an honest comparison.


In [8]:
FORECAST_ORIGIN_UTC = "08:30"  # on the day before the target day; used from notebook 04
